# 03 · Evaluación experimental del diseño

**Entregable 4** · Bases de Datos II · PanTEon Database v1.6.1

Rama: `feature/kmers-benchmark`

---

## Objetivo (guía, §4.4)

Comparar **al menos dos estrategias de almacenamiento** de los perfiles de *k*-mers y demostrar con mediciones propias cuál conviene y en qué escenario.

Pares de estrategias posibles (elegir una comparación y justificarla):

| Estrategia A | Estrategia B |
|---|---|
| Diccionario disperso (solo *k*-mers con conteo > 0) | Arreglo denso posicional en orden lexicográfico |
| Perfiles embebidos en el documento de la secuencia | Colección aparte `kmer_profiles` con referencia |
| Un documento por *k* | Un solo documento con los seis *k* |

## Métricas obligatorias para cada estrategia

- [ ] Tiempo de carga (misma máquina, mismo lote, repetir y reportar mediana).
- [ ] Espacio en disco con `db.collection.stats()` / `collStats` (incluir `storageSize`, `totalIndexSize` y `avgObjSize`).
- [ ] Tiempo de **3 consultas representativas**, con y sin índice, usando `explain("executionStats")`.

## Registro experimental

- [ ] Anotar hardware/entorno (Colab o local), versión de MongoDB y tamaño del lote.
- [ ] Fijar el conjunto de prueba (mismas secuencias para A y B).
- [ ] Documentar si se limpia la caché entre repeticiones y cómo.
- [ ] Añadir las filas correspondientes en `docs/decisiones.md` (sección 4).

## Salidas esperadas

- [ ] Tabla comparativa: estrategia × (tiempo de carga, MB en disco, tiempo de consulta con/sin índice).
- [ ] Gráficos de barras comparativos.
- [ ] Párrafo de discusión: qué estrategia conviene según el tipo de uso (secuencias cortas vs. largas, lectura vs. escritura).


In [5]:
from pymongo import MongoClient

# conexion local
cliente = MongoClient('mongodb://localhost:27017/')
db = cliente['panteon_db']
col_a = db['kmers']

print("--- BENCHMARK ESTRATEGIA A (1 doc x K) ---")

# peso y stats de la coleccion original
stats_a = db.command("collstats", "kmers")
print(f"Docs totales: {stats_a['count']}")
print(f"Peso en disco: {stats_a['storageSize'] / (1024*1024):.2f} MB")
print(f"Promedio por doc: {stats_a['avgObjSize']} bytes")

# queries de prueba
q1 = {"id_seq": "PDB00000001", "k": 6}
q2 = {"id_seq": "PumCon-1.11"}
q3 = {"k": 6, "valid_windows": {"$gt": 1500}}

# probar tiempos sin indices
print("\nTiempos SIN indices:")
print(f"Q1 (Puntual): {col_a.find(q1).explain()['executionStats']['executionTimeMillis']} ms")
print(f"Q2 (Perfil entero): {col_a.find(q2).explain()['executionStats']['executionTimeMillis']} ms")
print(f"Q3 (Filtro > 1500): {col_a.find(q3).explain()['executionStats']['executionTimeMillis']} ms")

# montamos los indices compuestos
print("\nCreando indices...")
col_a.create_index([("id_seq", 1), ("k", 1)])
col_a.create_index([("k", 1), ("valid_windows", 1)])

stats_a_post = db.command("collstats", "kmers")
print(f"Peso de los indices: {stats_a_post['totalIndexSize'] / (1024*1024):.2f} MB")

# probar tiempos con indices
print("\nTiempos CON indices:")
print(f"Q1: {col_a.find(q1).explain()['executionStats']['executionTimeMillis']} ms")
print(f"Q2: {col_a.find(q2).explain()['executionStats']['executionTimeMillis']} ms")
print(f"Q3: {col_a.find(q3).explain()['executionStats']['executionTimeMillis']} ms")


print("\n\n--- BENCHMARK ESTRATEGIA B (Docs anidados) ---")

col_b = db['kmers_estrategia_B']
col_b.drop() # limpiar por si ya existia

print("Agrupando kmers en la estrategia B, un momento...")
pipe_agrupar = [
    {"$group": {
        "_id": "$sequence_id",
        "id_seq": {"$first": "$id_seq"},
        "perfiles_k": {
            "$push": {
                "k": "$k",
                "valid_windows": "$valid_windows",
                "counts": "$counts"
            }
        }
    }},
    {"$out": "kmers_estrategia_B"}
]
col_a.aggregate(pipe_agrupar)

# revisar stats de la nueva coleccion
stats_b = db.command("collstats", "kmers_estrategia_B")
print(f"\nDocs totales: {stats_b['count']}")
print(f"Peso en disco: {stats_b['storageSize'] / (1024*1024):.2f} MB")
print(f"Promedio por doc: {stats_b['avgObjSize']} bytes")

# adaptamos las queries a la estructura nueva
q1_b = {"id_seq": "PDB00000001", "perfiles_k.k": 6}
q2_b = {"id_seq": "PumCon-1.11"}
q3_b = {"perfiles_k": {"$elemMatch": {"k": 6, "valid_windows": {"$gt": 1500}}}}

# tiempos sin indices
print("\nTiempos SIN indices:")
print(f"Q1 (Puntual): {col_b.find(q1_b).explain()['executionStats']['executionTimeMillis']} ms")
print(f"Q2 (Perfil entero): {col_b.find(q2_b).explain()['executionStats']['executionTimeMillis']} ms")
print(f"Q3 (Filtro > 1500): {col_b.find(q3_b).explain()['executionStats']['executionTimeMillis']} ms")

print("\nCreando indices...")
col_b.create_index([("id_seq", 1)])
col_b.create_index([("perfiles_k.k", 1), ("perfiles_k.valid_windows", 1)])

stats_b_post = db.command("collstats", "kmers_estrategia_B")
print(f"Peso de los indices: {stats_b_post['totalIndexSize'] / (1024*1024):.2f} MB")

# tiempos con indices
print("\nTiempos CON indices:")
print(f"Q1: {col_b.find(q1_b).explain()['executionStats']['executionTimeMillis']} ms")
print(f"Q2: {col_b.find(q2_b).explain()['executionStats']['executionTimeMillis']} ms")
print(f"Q3: {col_b.find(q3_b).explain()['executionStats']['executionTimeMillis']} ms")

--- BENCHMARK ESTRATEGIA A (1 doc x K) ---
Docs totales: 1439676
Peso en disco: 3469.56 MB
Promedio por doc: 5644 bytes

Tiempos SIN indices:
Q1 (Puntual): 0 ms
Q2 (Perfil entero): 0 ms
Q3 (Filtro > 1500): 569 ms

Creando indices...
Peso de los indices: 46.68 MB

Tiempos CON indices:
Q1: 0 ms
Q2: 0 ms
Q3: 555 ms


--- BENCHMARK ESTRATEGIA B (Docs anidados) ---
Agrupando kmers en la estrategia B, un momento...

Docs totales: 239946
Peso en disco: 3249.50 MB
Promedio por doc: 33553 bytes

Tiempos SIN indices:
Q1 (Puntual): 3182 ms
Q2 (Perfil entero): 514 ms
Q3 (Filtro > 1500): 535 ms

Creando indices...
Peso de los indices: 15.87 MB

Tiempos CON indices:
Q1: 0 ms
Q2: 0 ms
Q3: 1217 ms
